# 03 · Comparação de modelos (T1)

**Onde estamos no ML:** entregável 4. Aqui comparamos, na **mesma validação** do baseline, quatro modelos treinados por nós (**Regressão Logística**, **Árvore de Decisão**, **Random Forest** e **KNN**) contra a **lista fixa** (referência sem ML). Todos os experimentos estão registrados no MLflow (experimento `t1-manter-descartar`); este notebook lê as runs de lá.

| Item | Tipo |
| --- | --- |
| Lista fixa (`FUNCTION_WORDS`) | regra, sem ML (comportamento atual da API) |
| Regressão Logística, Árvore, Random Forest, KNN | **ML nosso** |
| Métrica principal | F1-macro; secundárias: ROC-AUC, PR-AUC, revocação por classe, tempo |

**Cuidados:** o teste final (10% das frases) segue intocado; tudo aqui é validação. O KNN foi treinado numa amostra de 200 mil palavras e com atributos reduzidos (TruncatedSVD, 50 componentes) porque vizinhos em milhares de colunas esparsas são caros; os demais usam as 1,3 milhão de palavras de ajuste.

In [1]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from textgloss import modeling, train
URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(URI); mlflow.set_experiment(train.EXPERIMENT)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
fit, val = modeling.train_validation(modeling.load_words())
y = val[modeling.TARGET].to_numpy()
runs = train.latest_runs()
scores = {n: np.load(mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{runs[n].info.run_id}/predicoes_validacao.npy", tracking_uri=URI)) for n in train.MODEL_NAMES}
preds = {n: (s >= 0.5).astype(int) for n, s in scores.items()}
preds["lista-fixa"] = modeling.fixed_list_predict(val)
val = val.assign(**{f"pred_{n}": p for n, p in preds.items()})
ORDEM = ["lista-fixa"] + train.MODEL_NAMES
print(f"validação: {len(val):,} palavras de {val.frase_id.nunique():,} frases · MLflow: {URI}")

validação: 145,918 palavras de 11,459 frases · MLflow: http://localhost:5000


## 1. Métricas

In [2]:
rows = []
for n in ORDEM:
    m = runs[n].data.metrics
    rows.append({"modelo": n, "f1_macro": m.get("val_f1_macro"), "roc_auc": m.get("val_roc_auc"), "pr_auc": m.get("val_pr_auc"),
                 "rec_descartar": m.get("val_recall_descartar"), "rec_manter": m.get("val_recall_manter"),
                 "f1_treino": m.get("treino_f1_macro"), "gap_treino_val": m.get("gap_f1_macro_treino_val"),
                 "treino_s": m.get("tempo_treino_s"), "predicao_s": m.get("tempo_predicao_val_s")})
tabela = pd.DataFrame(rows).set_index("modelo")
tabela.round(4)

,f1_macro,roc_auc,pr_auc,rec_descartar,rec_manter,f1_treino,gap_treino_val,treino_s,predicao_s
modelo,,,,,,,,,
lista-fixa,0.8221,0.8207,0.7640,0.7216,0.9197,NaN,NaN,NaN,NaN
regressao-logistica,0.8759,0.9442,0.9422,0.8243,0.9248,0.8770,0.0011,40.8371,0.4368
arvore-de-decisao,0.8703,0.9139,0.8835,0.8128,0.9247,0.8795,0.0092,86.3755,0.4403
random-forest,0.8524,0.9207,0.9112,0.7698,0.9315,0.8532,0.0008,98.3464,0.8129
knn,0.8465,0.8952,0.8669,0.7813,0.9088,0.9844,0.1379,4.4790,25.6048


*ROC-AUC e PR-AUC da lista fixa vêm de uma decisão binária, então não são comparáveis em pé de igualdade com as dos modelos.*

In [3]:
fig, ax = plt.subplots(figsize=(7, 3.2))
tabela["f1_macro"].sort_values().plot.barh(ax=ax, color="#4c78a8"); ax.set_xlim(0.75, 1.0); ax.set_xlabel("F1-macro (validação)")
for i, v in enumerate(tabela["f1_macro"].sort_values()): ax.text(v + 0.002, i, f"{v:.3f}", va="center")
plt.tight_layout()

## 2. Matrizes de confusão

In [4]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, n in zip(axes, ORDEM):
    ConfusionMatrixDisplay.from_predictions(y, preds[n], display_labels=modeling.CLASS_NAMES, cmap="Blues", ax=ax, colorbar=False, values_format="d")
    ax.set_title(f"{n}\nF1-macro {tabela.loc[n, 'f1_macro']:.3f}")
plt.tight_layout()

## 3. Curvas ROC e precisão-revocação

In [5]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.2))
for n, s in scores.items():
    fpr, tpr, _ = roc_curve(y, s); a.plot(fpr, tpr, label=f"{n} ({roc_auc_score(y, s):.3f})")
    p, r, _ = precision_recall_curve(y, s); b.plot(r, p, label=f"{n} ({average_precision_score(y, s):.3f})")
a.plot([0, 1], [0, 1], "--", color="gray"); a.set(xlabel="falso positivo", ylabel="verdadeiro positivo", title="ROC (AUC)")
b.set(xlabel="revocação", ylabel="precisão", title="Precisão-revocação (AP)"); a.legend(loc="lower right"); b.legend(loc="lower left")
plt.tight_layout()

## 4. Sobreajuste e custo
Uma diferença grande entre F1 de treino e de validação indica sobreajuste; uma diferença nula com F1 baixo indica modelo simples demais. O custo conta porque o classificador roda a cada frase na API.

In [6]:
tabela[["f1_treino", "f1_macro", "gap_treino_val", "treino_s", "predicao_s"]].dropna(how="all").round(3)

,f1_treino,f1_macro,gap_treino_val,treino_s,predicao_s
modelo,,,,,
lista-fixa,NaN,0.822,NaN,NaN,NaN
regressao-logistica,0.877,0.876,0.001,40.837,0.437
arvore-de-decisao,0.880,0.870,0.009,86.376,0.440
random-forest,0.853,0.852,0.001,98.346,0.813
knn,0.984,0.846,0.138,4.479,25.605


## 5. Onde cada modelo erra: por classe gramatical
F1-macro por UPOS (só classes com ≥1.000 palavras na validação). Mostra se o ganho do ML vem de onde esperamos (classes ambíguas) ou de onde a lista fixa já acertava.

In [7]:
def f1_por_upos(col):
    out = {}
    for upos, g in val.groupby("upos"):
        if len(g) >= 1000:
            out[upos] = f1_score(g[modeling.TARGET], g[col], average="macro")
    return pd.Series(out)
upos = pd.DataFrame({n: f1_por_upos(f"pred_{n}") for n in ORDEM})
upos["n"] = val.groupby("upos").size()
upos.sort_values("n", ascending=False).round(3)

,lista-fixa,regressao-logistica,arvore-de-decisao,random-forest,knn,n
NOUN,0.460,0.674,0.604,0.473,0.528,34052
ADP,0.649,0.708,0.697,0.483,0.596,27093
VERB,0.489,0.753,0.741,0.626,0.660,17671
PROPN,0.479,0.620,0.632,0.454,0.581,16387
DET,0.760,0.841,0.846,0.692,0.798,13571
ADJ,0.444,0.619,0.594,0.450,0.523,9653
PRON,0.478,0.798,0.823,0.815,0.792,6279
ADV,0.468,0.738,0.718,0.629,0.646,6057
CCONJ,0.498,0.496,0.588,0.474,0.537,4992
AUX,0.551,0.673,0.744,0.490,0.648,4918


In [8]:
ax = upos.drop(columns="n").sort_values("lista-fixa").plot.barh(figsize=(8, 7), width=0.8)
ax.set_xlabel("F1-macro"); ax.set_title("F1-macro por classe gramatical"); plt.tight_layout()

## 6. Erros do melhor modelo
Exemplos de falsos positivos (previu `MANTER`, o intérprete descartou) e falsos negativos (previu `DESCARTAR`, o intérprete manteve). Parte desses erros é **ruído de rótulo**, não erro do modelo.

In [9]:
melhor = tabela.drop(index="lista-fixa")["f1_macro"].idxmax()
col = f"pred_{melhor}"
print("melhor modelo:", melhor)
erros = val[val[col] != val[modeling.TARGET]]
print(f"{len(erros):,} erros em {len(val):,} palavras ({len(erros)/len(val):.1%})")
print("\nLemas mais errados (falso positivo = previu MANTER, o intérprete descartou):")
fp = erros[erros[col] == 1].groupby("lema").size().sort_values(ascending=False).head(12)
fn = erros[erros[col] == 0].groupby("lema").size().sort_values(ascending=False).head(12)
pd.concat({"falso_positivo": fp.reset_index(drop=True).to_frame("n").join(fp.index.to_series().reset_index(drop=True).rename("lema")),
           "falso_negativo": fn.reset_index(drop=True).to_frame("n").join(fn.index.to_series().reset_index(drop=True).rename("lema"))}, axis=1)

melhor modelo: regressao-logistica
17,943 erros em 145,918 palavras (12.3%)

Lemas mais errados (falso positivo = previu MANTER, o intérprete descartou):


falso_positivo         falso_negativo           
                n    lema              n       lema
0             292      eu            450       de o
1             212    mais            416         de
2             184     ter            377        que
3             137     seu            315         ou
4             114     ele            312        seu
5              84   fazer            194       como
6              80   dever            123          e
7              75     não             99  solicitar
8              64    após             90       isso
9              62   sobre             73        ter
10             56  quando             71         um
11             56  grande             66      entre

In [10]:
frases = pd.read_parquet(modeling.config.PROCESSED_DIR / "frases.parquet").set_index("frase_id")
amostra = erros.sample(8, random_state=1)
for _, r in amostra.iterrows():
    tipo = "previu MANTER, intérprete descartou" if r[col] == 1 else "previu DESCARTAR, intérprete manteve"
    print(f"[{r['palavra']}] ({r['lema']}/{r['upos']}) {tipo}\n  texto: {frases.loc[r['frase_id'], 'texto'][:110]}\n  glosa: {frases.loc[r['frase_id'], 'glosa'][:110]}\n")

[e] (e/CCONJ) previu DESCARTAR, intérprete manteve
  texto: O solicitante deverá realizar contato por telefone, e-mail, ofício ou presencial para obter informações sobre 
  glosa: SOLICITANTE DEVER&OBRIGAÇÃO REALIZAR CONTATO&TOQUE TELEFONE E_MAIL OFÍCIO RECUSAR OBTER INFORMAÇÃO SOBRE&ASSUN

[raramente] (raramente/ADV) previu MANTER, intérprete descartou
  texto: Por raramente se reunirem e não compartilharem de uma mesma opinião, os pintores pós-impressionistas são consi
  glosa: RARO REUNIR NÃO COMPARTILHAR MESMA OPINIÃO PINTOR PÓS IMPRESSIONISTA CONSIDERAR SOLITÁRIO ÚNICO UNIÃO ELES DEI

[marcante] (marcante/ADJ) previu MANTER, intérprete descartou
  texto: Com a presença de riachos, matas e montanhas, muitas atividades são pouco exploradas pelas pessoas, mas alguns
  glosa: PRESENTE&PRESENÇA RIACHO MATA MONTANHA MUITO ATIVIDADE POUCO EXPLORAR PESSOA ALGUM MORADOR VISITAR FAZER BOM U

[o] (ele/PRON) previu MANTER, intérprete descartou
  texto: Em outras situações que impliquem o tran

## 7. Os modelos erram nas mesmas palavras?
Se os erros coincidem, o limite está nos dados e nos atributos, não no tipo de modelo.

In [11]:
modelos = train.MODEL_NAMES
err = pd.DataFrame({n: (val[f"pred_{n}"] != val[modeling.TARGET]) for n in modelos})
print(f"erram todos os 4 modelos: {err.all(axis=1).mean():.1%} das palavras")
print(f"erra algum mas não todos: {(err.any(axis=1) & ~err.all(axis=1)).mean():.1%}")
print(f"nenhum erra: {(~err.any(axis=1)).mean():.1%}")
err.corr().round(2)

erram todos os 4 modelos: 7.6% das palavras
erra algum mas não todos: 13.1%
nenhum erra: 79.2%


,regressao-logistica,arvore-de-decisao,random-forest,knn
regressao-logistica,1.00,0.70,0.71,0.61
arvore-de-decisao,0.70,1.00,0.72,0.68
random-forest,0.71,0.72,1.00,0.76
knn,0.61,0.68,0.76,1.00


## 8. Análise crítica

**1. Todos os modelos treinados superam a lista fixa.** O F1-macro vai de 0,822 (lista) para 0,846–0,876. O ML vale a pena neste problema, mas o ganho é de **3 a 5 pontos**, não de dezenas.

**2. A Regressão Logística é a melhor** (F1-macro 0,876, ROC-AUC 0,944, PR-AUC 0,942), a mais barata (40 s de treino, 0,4 s para prever 146 mil palavras) e a única de que dá para ler o raciocínio pelos coeficientes. A **Árvore de Decisão** fica a 0,6 ponto (0,870), com a vantagem de produzir regras que se explicam em uma frase, mas com ROC-AUC menor (0,914): ela decide em degraus, e a regressão ordena melhor as palavras por confiança.

**3. O Random Forest ficou abaixo do esperado** (0,852), atrás até da árvore única. Provável causa: com `max_features="sqrt"` sobre mais de 2.000 colunas one-hot esparsas, cada árvore enxerga poucos atributos úteis por divisão, e a profundidade e a folha mínima foram fixadas sem busca. Isto **não é uma conclusão sobre florestas em geral**: é um resultado sem ajuste de hiperparâmetros, que a fase 5 (Optuna) pode mudar.

**4. O KNN sobreajusta e é caro.** F1-macro de 0,984 no treino contra 0,846 na validação (diferença de 0,138, a única relevante), e 25 s para prever 146 mil palavras, 60 vezes mais que os outros. Foi treinado numa amostra de 200 mil palavras com atributos reduzidos a 50 componentes, o que também limita o resultado. Não serve para a API.

**5. Onde o ML ganha.** Por classe gramatical (seção 5), os maiores saltos sobre a lista fixa estão em **pronomes** (0,48 → 0,80), **verbos** (0,49 → 0,75), **advérbios** (0,47 → 0,74) e **substantivos/adjetivos**. São exatamente as classes em que a lista fixa não tem como acertar, porque só conhece palavras funcionais. Ressalva: o F1-macro dentro de uma classe gramatical é sensível ao desbalanceamento interno (quase todo substantivo é `MANTER`), então estes valores servem para comparar modelos entre si, não como medida absoluta.

**6. Os erros restantes são em grande parte ambiguidade e ruído.** Os quatro modelos erram juntos em 7,6% das palavras e erram em coisas diferentes em 13,1%, o que mostra que ainda há espaço entre modelos, mas que existe um piso comum nos dados. Os lemas mais errados da regressão são ambíguos de verdade (`eu`, `mais`, `ter`, `seu`, `ele`, `de`, `que`, `ou`): os intérpretes ora mantêm, ora descartam, e os atributos atuais não carregam o contexto necessário (só a classe gramatical dos vizinhos, não os lemas deles). Há também ruído de rótulo: `solicitar` aparece como falso negativo porque o modelo aprendeu, de glosas que usam outro sinal, que ele costuma ser descartado.

**7. O que não dá para concluir.**
- A validação é um **único sorteio** de frases; a estabilidade entre dobras só vem na fase 6.
- Os modelos usaram **hiperparâmetros fixos**; a ordem Árvore/Floresta pode mudar com a busca.
- As métricas medem proximidade às glosas dos intérpretes, com **ruído de rótulo ainda não medido** (conferência manual das 200 palavras pendente).
- O KNN usou uma amostra menor que os demais.

**8. Próximos passos sugeridos.** (a) Fase 5 (Optuna): a busca fica mais informativa sobre a **Árvore ou o Random Forest**, que têm muitos hiperparâmetros e estão longe do seu limite, do que sobre a regressão, que só tem `C`. Vale decidir isso explicitamente. (b) Acrescentar os **lemas dos vizinhos** como atributos pode atacar a ambiguidade do item 6. (c) Fazer a conferência manual de rótulos para saber o teto real de F1.